In [ ]:
# =============================================================================
# CELL 1: INSTALL DEPENDENCIES + BUILD BETTI-MATCHING-3D
# =============================================================================
import subprocess, sys, os, types
from pathlib import Path
import importlib.util as _ilu

WHEELS_DIR = Path("/kaggle/input/datasets/sohier/vesuvius-metric-resources/wheels")
METRICS_DIR = Path("/kaggle/input/datasets/sohier/vesuvius-metric-resources/topological-metrics-kaggle")

for alt in [Path("/kaggle/input/topological-metrics-kaggle"),
            Path("/kaggle/input/vesuvius-metrics"),
            Path("/kaggle/input/datasets/rajbalabala/topological-metrics-kaggle")]:
    if not WHEELS_DIR.exists() and (alt / "wheels").exists(): WHEELS_DIR = alt / "wheels"
    if not METRICS_DIR.exists() and (alt / "src" / "topometrics").exists(): METRICS_DIR = alt

print(f"Wheels: {WHEELS_DIR} ({WHEELS_DIR.exists()})")
print(f"Metrics: {METRICS_DIR} ({METRICS_DIR.exists()})")

# Install wheels (skip incompatible)
if WHEELS_DIR.exists():
    for w in sorted(WHEELS_DIR.glob("*.whl")):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-q", str(w)],
                           capture_output=True, text=True)
        if r.returncode != 0: print(f"  SKIP: {w.name}")

# PyPI fallback
for mod, pip in {"surface_distance":"surface-distance","cc3d":"connected-components-3d",
                 "tifffile":"tifffile","imagecodecs":"imagecodecs"}.items():
    try: __import__(mod)
    except ImportError:
        subprocess.run([sys.executable,"-m","pip","install","-q",pip], check=False)

subprocess.run([sys.executable,"-m","pip","install","-q","pybind11[global]","cmake"], capture_output=True, check=False)

# Build Betti-Matching-3D
BETTI_SRC = METRICS_DIR / "external" / "Betti-Matching-3D"
BUILD_DIR = Path("/kaggle/working/betti_build")
if BETTI_SRC.exists() and not list(BUILD_DIR.rglob("betti_matching*")):
    BUILD_DIR.mkdir(parents=True, exist_ok=True)
    import sysconfig
    py_inc = sysconfig.get_config_var("INCLUDEPY") or sysconfig.get_paths().get("include","")
    cmake_args = ["cmake","-S",str(BETTI_SRC),"-B",str(BUILD_DIR),
                  f"-DPython_EXECUTABLE={sys.executable}",f"-DPython_INCLUDE_DIR={py_inc}",
                  "-DPYBIND11_FINDPYTHON=ON"]
    py_libdir = sysconfig.get_config_var("LIBDIR") or ""
    ldver = sysconfig.get_config_var("LDVERSION") or sysconfig.get_config_var("VERSION")
    for ext in [".so",".a"]:
        c = os.path.join(py_libdir, f"libpython{ldver}{ext}")
        if os.path.exists(c): cmake_args.append(f"-DPython_LIBRARY={c}"); break
    print("Building Betti-Matching-3D...")
    r1 = subprocess.run(cmake_args, capture_output=True, text=True)
    if r1.returncode == 0:
        r2 = subprocess.run(["cmake","--build",str(BUILD_DIR),"--parallel"], capture_output=True, text=True)
        if r2.returncode == 0: print("  Built!")
        else: print(f"  Build failed: {r2.stderr[-400:]}")
    else: print(f"  CMake failed: {r1.stderr[-400:]}")

# Add topometrics to path
TOPO_SRC = METRICS_DIR / "src"
if TOPO_SRC.exists() and str(TOPO_SRC) not in sys.path:
    sys.path.insert(0, str(TOPO_SRC))

# Pre-import betti_matching
betti_mod = None
if BUILD_DIR.exists():
    for d in [BUILD_DIR] + [p.parent for p in BUILD_DIR.rglob("*.so")]:
        if str(d) not in sys.path: sys.path.insert(0, str(d))
    try:
        import betti_matching as _bm; betti_mod = _bm; print("  betti_matching OK")
    except ImportError: pass
    if betti_mod is None:
        for so in sorted(BUILD_DIR.rglob("*.so")):
            if "betti" in so.name.lower():
                try:
                    spec = _ilu.spec_from_file_location("betti_matching", str(so))
                    if spec and spec.loader:
                        betti_mod = _ilu.module_from_spec(spec)
                        spec.loader.exec_module(betti_mod)
                        sys.modules["betti_matching"] = betti_mod
                        print(f"  betti_matching from {so}")
                        break
                except: pass

if betti_mod is not None:
    fake = types.ModuleType("topometrics._bm_loader")
    fake.load_betti_matching = lambda: betti_mod
    sys.modules["topometrics._bm_loader"] = fake

HAVE_FULL_METRICS = False
try:
    from topometrics import compute_leaderboard_score
    HAVE_FULL_METRICS = True; print("topometrics OK!")
except ImportError as e:
    print(f"topometrics failed: {e}")

print(f"Python {sys.version.split()[0]}, Full metrics: {HAVE_FULL_METRICS}")


In [ ]:
# =============================================================================
# CELL 2: IMPORTS + CONFIG + FOLDS + ARCHITECTURE
# =============================================================================
import os, gc, json, time, random, warnings, traceback
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field
from itertools import product

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.ndimage import (binary_fill_holes, binary_closing, binary_opening,
                            label as ndimage_label, generate_binary_structure)
from skimage.morphology import remove_small_objects
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIG
# =============================================================================
@dataclass
class CVConfig:
    DATA_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-surface-detection")
    FOLD0_BEST: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/best_model.pth")
    FOLD0_CKPT_A: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_460.pth")
    FOLD0_CKPT_B: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_440.pth")
    FOLD0_CKPT_C: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_420.pth")
    FOLD1_BEST: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/best_model.pth")
    FOLD1_CKPT_A: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_100.pth")
    FOLD1_CKPT_B: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_120.pth")
    FOLD1_CKPT_C: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_110.pth")
    FOLD0_FEATURES: List[int] = field(default_factory=lambda: [32,64,128,256,320,320])
    FOLD0_N_BLOCKS: List[int] = field(default_factory=lambda: [1,2,3,4,6,6])
    FOLD1_FEATURES: List[int] = field(default_factory=lambda: [32,64,128,256,320,320])
    FOLD1_N_BLOCKS: List[int] = field(default_factory=lambda: [1,2,3,4,6,6])
    N_FOLDS: int = 3; CV_SEED: int = 42
    PATCH_SIZE: Tuple[int,int,int] = (192,192,192); OVERLAP: float = 0.5
    MAX_EVAL_SAMPLES: int = 15  # 15 for speed — enough signal
    OUTPUT_DIR: Path = Path("/kaggle/working/cv_eval")
    PRED_DIR: Path = Path("/kaggle/working/cv_eval/predictions")
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CVConfig()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.PRED_DIR.mkdir(parents=True, exist_ok=True)

for c in [Path("/kaggle/input/datasets/rajbalabala/3d-volume-training-data"),
          Path("/kaggle/input/datasets/asharamkanderiwal/aimo-dataset"),
          Path("/kaggle/input/3d-volume-training-data"),
          Path("/kaggle/input/vesuvius-challenge-surface-detection")]:
    if (c / "train.csv").exists(): cfg.DATA_ROOT = c; break

N_GPUS = min(torch.cuda.device_count(), 1) if torch.cuda.is_available() else 0
print(f"Data: {cfg.DATA_ROOT} | GPU: {torch.cuda.get_device_name(0) if N_GPUS else 'CPU'} | Eval: {cfg.MAX_EVAL_SAMPLES} vols")

FOLD_MODELS = {
    0: {'paths':[cfg.FOLD0_BEST,cfg.FOLD0_CKPT_A,cfg.FOLD0_CKPT_B,cfg.FOLD0_CKPT_C],
        'names':['fold0_best','fold0_ckptA','fold0_ckptB','fold0_ckptC'],
        'features':cfg.FOLD0_FEATURES,'n_blocks':cfg.FOLD0_N_BLOCKS},
    1: {'paths':[cfg.FOLD1_BEST,cfg.FOLD1_CKPT_A,cfg.FOLD1_CKPT_B,cfg.FOLD1_CKPT_C],
        'names':['fold1_best','fold1_ckptA','fold1_ckptB','fold1_ckptC'],
        'features':cfg.FOLD1_FEATURES,'n_blocks':cfg.FOLD1_N_BLOCKS},
}

# =============================================================================
# FOLD SPLITS
# =============================================================================
def make_folds(csv_path, img_dir, lbl_dir, n_splits=3, seed=42):
    df = pd.read_csv(csv_path)
    valid = df['id'].apply(lambda x: (img_dir/f"{x}.tif").exists() and (lbl_dir/f"{x}.tif").exists())
    df = df[valid].reset_index(drop=True)
    strat = df['scroll_id'].values if 'scroll_id' in df.columns else df['id'].apply(lambda x:x.split('_')[0] if '_' in x else 'unk').values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for fold, (ti, vi) in enumerate(skf.split(df['id'], strat)):
        splits.append((df.iloc[ti]['id'].tolist(), df.iloc[vi]['id'].tolist()))
        print(f"  Fold {fold}: train={len(ti)}, val={len(vi)}")
    return splits

FOLD_SPLITS = []
if (cfg.DATA_ROOT/"train.csv").exists():
    FOLD_SPLITS = make_folds(cfg.DATA_ROOT/"train.csv", cfg.DATA_ROOT/"train_images",
                              cfg.DATA_ROOT/"train_labels", cfg.N_FOLDS, cfg.CV_SEED)

# =============================================================================
# MODEL ARCHITECTURE
# =============================================================================
def get_num_groups(ch, mx=32):
    for g in [mx,16,8,4,2,1]:
        if ch % g == 0: return g
    return 1

class HybridConv3d(nn.Module):
    def __init__(s, ic, oc):
        super().__init__(); mc=oc//2
        s.conv_xy=nn.Conv3d(ic,mc,kernel_size=(1,3,3),padding=(0,1,1),bias=False)
        s.conv_z=nn.Conv3d(ic,oc-mc,kernel_size=(3,1,1),padding=(1,0,0),bias=False)
        s.norm=nn.GroupNorm(get_num_groups(oc),oc); s.act=nn.LeakyReLU(0.01,True)
    def forward(s,x): return s.act(s.norm(torch.cat([s.conv_xy(x),s.conv_z(x)],1)))

class ConvBlock(nn.Module):
    def __init__(s,ic,oc,hyb=False):
        super().__init__()
        s.conv = HybridConv3d(ic,oc) if hyb else nn.Sequential(
            nn.Conv3d(ic,oc,3,padding=1,bias=False),nn.GroupNorm(get_num_groups(oc),oc),nn.LeakyReLU(0.01,True))
    def forward(s,x): return s.conv(x)

class MultiScaleResBlock(nn.Module):
    def __init__(s,ch,scales=2,hyb=False):
        super().__init__(); s.scales=scales; s.width=ch//scales
        s.convs=nn.ModuleList([ConvBlock(s.width,s.width,hyb) for _ in range(scales-1)])
        s.norm=nn.GroupNorm(get_num_groups(ch),ch)
    def forward(s,x):
        sp=torch.chunk(x,s.scales,1); o=[sp[0]]
        for i,c in enumerate(s.convs): o.append(c(sp[i+1]+o[-1]) if i>0 else c(sp[i+1]))
        return x+s.norm(torch.cat(o,1))

class AttentionBlock(nn.Module):
    def __init__(s,ch,r=4):
        super().__init__(); s.gap=nn.AdaptiveAvgPool3d(1)
        s.fc1=nn.Linear(ch,max(ch//r,8)); s.fc2=nn.Linear(max(ch//r,8),ch)
        s.cs=nn.Conv3d(2,1,7,padding=3,bias=False)
    def forward(s,x):
        b,c=x.shape[:2]
        ca=torch.sigmoid(s.fc2(F.relu(s.fc1(s.gap(x).view(b,c))))).view(b,c,1,1,1); xc=x*ca
        sa=torch.sigmoid(s.cs(torch.cat([xc.mean(1,True),xc.max(1,True)[0]],1)))
        return xc*sa

class SurfaceRefinementBlock(nn.Module):
    TILE_DEPTH=32
    def __init__(s,ic,oc):
        super().__init__(); s.edge_conv=nn.Conv3d(ic,ic,3,padding=1,bias=False)
        s.refine=nn.Sequential(nn.Conv3d(ic*2,oc,3,padding=1,bias=False),nn.GroupNorm(get_num_groups(oc),oc),
            nn.LeakyReLU(0.01,True),nn.Conv3d(oc,oc,3,padding=1,bias=False),nn.GroupNorm(get_num_groups(oc),oc),nn.LeakyReLU(0.01,True))
    def _forward_tiled(s,x):
        B,C,D,H,W=x.shape; tile=s.TILE_DEPTH; pad=1; out=[]
        for ds in range(0,D,tile):
            de=min(ds+tile,D); dsp=max(0,ds-pad); dep=min(D,de+pad)
            xt=x[:,:,dsp:dep,:,:]; et=torch.abs(s.edge_conv(xt))
            ot=s.refine(torch.cat([xt,et],1)); vs=ds-dsp; ve=vs+(de-ds)
            out.append(ot[:,:,vs:ve,:,:]); del xt,et,ot
        return torch.cat(out,2)
    def forward(s,x):
        if x.shape[2]<=s.TILE_DEPTH: return s.refine(torch.cat([x,torch.abs(s.edge_conv(x))],1))
        return torch.cat([s._forward_tiled(x[i:i+1]) for i in range(x.shape[0])],0)

class TopoPreservingUNet3D(nn.Module):
    def __init__(s,in_ch=1,out_ch=1,features=None,n_blocks=None,use_attention=True,use_hybrid_conv=True,use_surface_refinement=True):
        super().__init__()
        features=features or [32,64,128,256,320,320]; n_blocks=n_blocks or [1,2,3,4,6,6]; s.features=features
        s.encoders=nn.ModuleList(); s.attentions=nn.ModuleList(); s.pools=nn.ModuleList()
        for i,(f,nb) in enumerate(zip(features,n_blocks)):
            ic=in_ch if i==0 else features[i-1]
            layers=[ConvBlock(ic,f,hyb=use_hybrid_conv and i>0)]
            layers.extend([MultiScaleResBlock(f,2,use_hybrid_conv) for _ in range(nb)])
            s.encoders.append(nn.Sequential(*layers))
            s.attentions.append(AttentionBlock(f) if use_attention and i>=2 else nn.Identity())
            if i<len(features)-1: s.pools.append(nn.Conv3d(f,f,2,stride=2))
        s.ups=nn.ModuleList(); s.dec_convs=nn.ModuleList()
        for i in range(len(features)-2,-1,-1):
            s.ups.append(nn.ConvTranspose3d(features[i+1],features[i],2,stride=2))
            if use_surface_refinement and i==0: s.dec_convs.append(SurfaceRefinementBlock(features[i]*2,features[i]))
            else: s.dec_convs.append(nn.Sequential(ConvBlock(features[i]*2,features[i],use_hybrid_conv),MultiScaleResBlock(features[i],2,use_hybrid_conv)))
        s.final=nn.Conv3d(features[0],out_ch,1)
    def forward(s,x):
        ef=[]
        for i,(e,a) in enumerate(zip(s.encoders,s.attentions)):
            x=a(e(x)); ef.append(x)
            if i<len(s.pools): x=s.pools[i](x)
        ef=ef[::-1]; x=ef[0]
        for i,(u,d) in enumerate(zip(s.ups,s.dec_convs)):
            x=u(x); sk=ef[i+1]
            if x.shape[2:]!=sk.shape[2:]: x=F.interpolate(x,size=sk.shape[2:],mode='trilinear',align_corners=False)
            x=d(torch.cat([x,sk],1))
        return s.final(x)

def load_model_from_checkpoint(path, features, n_blocks, device='cpu'):
    m = TopoPreservingUNet3D(features=features, n_blocks=n_blocks)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    st = ckpt.get('model_state_dict', ckpt)
    st = {k.replace('_orig_mod.','').replace('module.',''): v for k,v in st.items()}
    m.load_state_dict(st, strict=False)
    ep, dice = ckpt.get('epoch','?'), ckpt.get('best_dice',0)
    del ckpt, st; gc.collect(); m.eval()
    return m, ep, dice

m = TopoPreservingUNet3D(features=cfg.FOLD0_FEATURES, n_blocks=cfg.FOLD0_N_BLOCKS)
print(f"Model: {sum(p.numel() for p in m.parameters())/1e6:.1f}M params"); del m


In [ ]:
# =============================================================================
# CELL 3: INFERENCE (H100 BATCHED) + METRICS + POSTPROCESSING
# =============================================================================

def robust_zscore_normalize(img, lp=0.5, up=99.5):
    lo, hi = np.percentile(img, lp), np.percentile(img, up)
    c = np.clip(img, lo, hi); return ((c - c.mean()) / (c.std() + 1e-8)).astype(np.float32)

def create_gaussian_weight(ps, sigma=0.125):
    d,h,w = ps
    gz=np.exp(-0.5*((np.arange(d)-d/2)/(d*sigma))**2)
    gy=np.exp(-0.5*((np.arange(h)-h/2)/(h*sigma))**2)
    gx=np.exp(-0.5*((np.arange(w)-w/2)/(w*sigma))**2)
    return (gz[:,None,None]*gy[None,:,None]*gx[None,None,:]).astype(np.float32)

def get_patch_positions(vs, ps, ov=0.5):
    D,H,W=vs; pd,ph,pw=ps
    sd,sh,sw=[max(1,int(p*(1-ov))) for p in ps]
    def _p(t,p,s):
        if t<=p: return [0]
        pos=list(range(0,t-p+1,s));
        if (t-p) not in pos: pos.append(t-p)
        return pos
    return [(z,y,x) for z in _p(D,pd,sd) for y in _p(H,ph,sh) for x in _p(W,pw,sw)]

@torch.no_grad()
def sliding_window_inference(model, vol, patch_size, overlap=0.5, device="cuda", verbose=True):
    """BATCHED sliding window — feeds multiple patches at once on H100."""
    model.eval(); D,H,W=vol.shape; pd,ph,pw=patch_size
    pad_d,pad_h,pad_w=max(0,pd-D),max(0,ph-H),max(0,pw-W)
    if pad_d>0 or pad_h>0 or pad_w>0:
        vol=np.pad(vol,((0,pad_d),(0,pad_h),(0,pad_w)),mode='reflect'); D,H,W=vol.shape
    pred_sum=np.zeros((D,H,W),np.float32); wt_sum=np.zeros((D,H,W),np.float32)
    gauss=create_gaussian_weight(patch_size)
    positions=get_patch_positions((D,H,W),patch_size,overlap)
    BATCH=8  # 8×192³×2B ≈ 0.5GB — trivial for H100
    if verbose: print(f"    {len(positions)} patches, batch={BATCH}")
    for bs in range(0,len(positions),BATCH):
        bp=positions[bs:bs+BATCH]; patches=[]
        for z,y,x in bp:
            p=vol[z:z+pd,y:y+ph,x:x+pw]
            if p.shape!=(pd,ph,pw): p=np.pad(p,[(0,max(0,s-a)) for a,s in zip(p.shape,(pd,ph,pw))],mode='reflect')
            patches.append(p)
        bt=torch.from_numpy(np.stack(patches)[:,None]).to(device).half()
        with torch.amp.autocast('cuda',dtype=torch.float16): preds=torch.sigmoid(model(bt))
        pn=preds.float().cpu().numpy()
        for i,(z,y,x) in enumerate(bp):
            pred_sum[z:z+pd,y:y+ph,x:x+pw]+=pn[i,0]*gauss; wt_sum[z:z+pd,y:y+ph,x:x+pw]+=gauss
        del bt,preds,pn
    torch.cuda.empty_cache()
    r=pred_sum/np.maximum(wt_sum,1e-8)
    return r[:D-pad_d or D,:H-pad_h or H,:W-pad_w or W]

# =============================================================================
# COMPETITION METRIC
# =============================================================================
def compute_competition_score(pred, gt, ignore_label=2):
    if HAVE_FULL_METRICS:
        rpt = compute_leaderboard_score(predictions=pred, labels=gt, dims=(0,1,2),
            spacing=(1.,1.,1.), surface_tolerance=2.0, voi_connectivity=26,
            combine_weights=(0.3,0.35,0.35), fg_threshold=None, ignore_label=ignore_label)
        return {'lb_score':rpt.score,'toposcore':rpt.topo.toposcore,
                'surface_dice':rpt.surface_dice,'voi_score':rpt.voi.voi_score,
                'voi_split':rpt.voi.voi_split,'voi_merge':rpt.voi.voi_merge}
    else:
        import surface_distance as sd; import cc3d as _cc
        ign=(gt==ignore_label); pr=pred.copy(); g=(gt==1).astype(np.uint8); pr[ign]=0; g[ign]=0
        if not g.any() and not pr.any(): sdice=1.0
        elif g.any()^pr.any(): sdice=0.0
        else: sdice=float(sd.compute_surface_dice_at_tolerance(sd.compute_surface_distances(g.astype(bool),pr.astype(bool),(1,1,1)),2.0))
        gl=_cc.connected_components(g,connectivity=26); pl=_cc.connected_components(pr,connectivity=26)
        u=(gl>0)|(pl>0)
        if u.any():
            from skimage.metrics import variation_of_information
            vs,vm=variation_of_information(gl[u],pl[u]); voi_score=1.0/(1.0+vs+vm)
        else: vs=vm=0.0; voi_score=1.0
        lb=0.30*0.5+0.35*sdice+0.35*voi_score
        return {'lb_score':lb,'toposcore':0.5,'surface_dice':sdice,'voi_score':voi_score,'voi_split':vs,'voi_merge':vm}

# =============================================================================
# POST-PROCESSING
# =============================================================================
def count_components(mask):
    _,n=ndimage_label(mask,generate_binary_structure(3,3)); return n

def remove_small_components(mask, min_size=50):
    s=generate_binary_structure(3,3); lab,n=ndimage_label(mask,s)
    if n==0: return mask
    sz=np.bincount(lab.ravel()); sm=sz<min_size; sm[0]=False; r=mask.copy(); r[sm[lab]]=0; return r

def topology_safe_op(mask, op, name=""):
    nb=count_components(mask); r=op(mask)
    return mask if count_components(r)<nb else r

def slicewise_hole_fill(mask):
    f=mask.copy()
    for ax in range(3):
        for i in range(mask.shape[ax]):
            if ax==0: f[i]=binary_fill_holes(f[i])
            elif ax==1: f[:,i,:]=binary_fill_holes(f[:,i,:])
            else: f[:,:,i]=binary_fill_holes(f[:,:,i])
    return f

def slicewise_morphology(mask, op='close', it=1):
    r=mask.copy(); s2=generate_binary_structure(2,1)
    fn=binary_closing if op=='close' else binary_opening
    for ax in range(3):
        for i in range(mask.shape[ax]):
            if ax==0: r[i]=fn(r[i],s2,iterations=it)
            elif ax==1: r[:,i,:]=fn(r[:,i,:],s2,iterations=it)
            else: r[:,:,i]=fn(r[:,:,i],s2,iterations=it)
    return r

def postprocess(prob, threshold=0.5, min_size=50,
                do_3d_close=True, do_3d_fill=True,
                do_slice_close=True, do_slice_fill=True, do_slice_open=True):
    mask=(prob>threshold).astype(np.uint8)
    if mask.sum()==0: return mask
    mask=remove_small_components(mask, min_size)
    s3=generate_binary_structure(3,3)
    if do_3d_close: mask=topology_safe_op(mask,lambda m:binary_closing(m,s3,1).astype(np.uint8))
    if do_3d_fill: mask=topology_safe_op(mask,lambda m:binary_fill_holes(m).astype(np.uint8))
    if do_slice_close: mask=topology_safe_op(mask,lambda m:slicewise_morphology(m,'close',1))
    if do_slice_fill: mask=topology_safe_op(mask,slicewise_hole_fill)
    if do_3d_fill: mask=topology_safe_op(mask,lambda m:binary_fill_holes(m).astype(np.uint8))
    if do_slice_open: mask=topology_safe_op(mask,lambda m:slicewise_morphology(m,'open',1))
    return remove_small_components(mask, min_size)

print("Inference (batched) + Metrics + PostProcessing ready")


In [ ]:
# =============================================================================
# CELL 4: OOF PREDICTIONS — H100 FAST (single GPU, batched inference)
# =============================================================================

def generate_oof_predictions():
    images_dir = cfg.DATA_ROOT / "train_images"
    labels_dir = cfg.DATA_ROOT / "train_labels"
    
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Eval: {cfg.MAX_EVAL_SAMPLES} vols/fold, overlap={cfg.OVERLAP}")
    
    all_scores = []
    for fold_idx in [0, 1]:
        fi = FOLD_MODELS[fold_idx]; _, val_ids = FOLD_SPLITS[fold_idx]
        features, n_blocks = fi['features'], fi['n_blocks']
        
        valid_ids = [v for v in val_ids if (images_dir/f"{v}.tif").exists() and (labels_dir/f"{v}.tif").exists()]
        rng = np.random.RandomState(cfg.CV_SEED + fold_idx)
        eval_ids = sorted(rng.choice(valid_ids, min(cfg.MAX_EVAL_SAMPLES, len(valid_ids)), replace=False).tolist()) if cfg.MAX_EVAL_SAMPLES>0 and len(valid_ids)>cfg.MAX_EVAL_SAMPLES else sorted(valid_ids)
        
        print(f"\n{'='*60}\nFOLD {fold_idx}: {len(eval_ids)} volumes\n{'='*60}")
        swa_acc, swa_cnt = {}, {}
        
        for mi, (ckpt, mname) in enumerate(zip(fi['paths'], fi['names'])):
            if not Path(ckpt).exists(): print(f"  SKIP {mname}"); continue
            
            model, ep, dice = load_model_from_checkpoint(ckpt, features, n_blocks)
            model = model.to('cuda:0').half()
            torch.cuda.synchronize()
            print(f"\n  --- {mname} (ep={ep}, dice={dice:.4f}) ---")
            
            t0 = time.time()
            for vi, vid in enumerate(eval_ids):
                raw = tifffile.imread(str(images_dir/f"{vid}.tif")).astype(np.float32)
                norm = robust_zscore_normalize(raw); del raw
                prob = sliding_window_inference(model, norm, cfg.PATCH_SIZE, cfg.OVERLAP, 'cuda:0', verbose=False)
                del norm; torch.cuda.empty_cache()
                
                gt = tifffile.imread(str(labels_dir/f"{vid}.tif")).astype(np.uint8)
                mask = postprocess(prob, 0.5, 50)
                scores = compute_competition_score(mask, gt, ignore_label=2)
                del mask, gt
                
                if vid in swa_acc: swa_acc[vid] += prob
                else: swa_acc[vid] = prob.copy()
                swa_cnt[vid] = swa_cnt.get(vid, 0) + 1
                del prob; gc.collect()
                
                all_scores.append({'fold':fold_idx,'vol_id':vid,'model':mname,**scores})
                elapsed = time.time()-t0; rate=(vi+1)/elapsed*60
                print(f"    [{vi+1}/{len(eval_ids)}] {vid}: LB={scores['lb_score']:.4f} ({rate:.1f} v/m)")
            
            del model; torch.cuda.empty_cache(); gc.collect()
            print(f"  {mname}: {time.time()-t0:.0f}s")
        
        # Save SWA
        swa_dir = cfg.PRED_DIR / f"fold{fold_idx}_swa"
        swa_dir.mkdir(parents=True, exist_ok=True)
        for vid in sorted(swa_acc):
            sp = swa_acc[vid] / swa_cnt[vid]
            np.save(str(swa_dir/f"{vid}.npy"), sp.astype(np.float16))
            gt = tifffile.imread(str(labels_dir/f"{vid}.tif")).astype(np.uint8)
            mask = postprocess(sp, 0.5, 50)
            sc = compute_competition_score(mask, gt, ignore_label=2)
            all_scores.append({'fold':fold_idx,'vol_id':vid,'model':f'fold{fold_idx}_swa',**sc})
            print(f"    SWA {vid}: LB={sc['lb_score']:.4f}")
            del sp, mask, gt
        del swa_acc, swa_cnt; gc.collect()
        
        fdf = pd.DataFrame([s for s in all_scores if s['fold']==fold_idx])
        if len(fdf):
            mm = fdf.groupby('model')['lb_score'].mean().sort_values(ascending=False)
            print(f"\n  FOLD {fold_idx} RANKING:"); [print(f"    {m:<20s}: {lb:.4f}{'★' if 'swa' in m else ''}") for m,lb in mm.items()]
    
    df = pd.DataFrame(all_scores)
    df.to_csv(cfg.OUTPUT_DIR/"model_comparison.csv", index=False)
    print(f"\nSaved: {cfg.OUTPUT_DIR/'model_comparison.csv'} ({len(df)} rows)")
    return df

model_df = generate_oof_predictions()


In [ ]:
# =============================================================================
# CELL 5: THRESHOLD SWEEP — fast serial (~5 min)
# =============================================================================
def sweep_thresholds():
    labels_dir = cfg.DATA_ROOT / "train_labels"
    thresholds = [0.35, 0.40, 0.45, 0.50, 0.55]
    min_sizes = [50, 100]
    results = []; t0 = time.time(); vc = 0
    
    for fold_idx in [0, 1]:
        swa_dir = cfg.PRED_DIR / f"fold{fold_idx}_swa"
        if not swa_dir.exists(): continue
        npy_files = sorted(swa_dir.glob("*.npy"))
        
        for vi, npy in enumerate(npy_files):
            vid = npy.stem; gt_path = labels_dir / f"{vid}.tif"
            if not gt_path.exists(): continue
            prob = np.load(str(npy)).astype(np.float32)
            gt = tifffile.imread(str(gt_path)).astype(np.uint8)
            
            best_lb, best_t, best_ms = 0, 0, 0
            for t in thresholds:
                for ms in min_sizes:
                    mask = postprocess(prob, t, ms); scores = compute_competition_score(mask, gt, 2)
                    results.append({'fold':fold_idx,'vol_id':vid,'threshold':round(t,2),'min_size':ms,**scores})
                    if scores['lb_score'] > best_lb: best_lb,best_t,best_ms = scores['lb_score'],t,ms
                    del mask
            del prob, gt; gc.collect(); vc += 1
            eta = (time.time()-t0)/vc*(len(npy_files)*2-vc)
            print(f"  [{vc}] f{fold_idx} {vid}: best T={best_t:.2f} ms={best_ms} LB={best_lb:.4f} ({eta:.0f}s left)")
    
    df = pd.DataFrame(results); df.to_csv(cfg.OUTPUT_DIR/"threshold_sweep.csv", index=False)
    cm = df.groupby(['threshold','min_size'])['lb_score'].mean().sort_values(ascending=False)
    print(f"\n{'='*50}")
    for (t,ms),lb in cm.items(): print(f"  T={t:.2f} ms={ms:3d}: LB={lb:.4f}{'★' if (t,ms)==cm.index[0] else ''}")
    bt,bms = cm.index[0]; print(f"\n★ BEST: threshold={bt}, min_size={bms}, LB={cm.iloc[0]:.4f}")
    return df

threshold_results = sweep_thresholds()


In [ ]:
# =============================================================================
# CELL 6: QUICK PP TEST — 4 combos (~3 min)
# =============================================================================
def quick_pp():
    labels_dir = cfg.DATA_ROOT / "train_labels"
    bt, bms = 0.50, 50
    if 'threshold_results' in dir() and len(threshold_results) > 0:
        cm = threshold_results.groupby(['threshold','min_size'])['lb_score'].mean()
        bt, bms = cm.idxmax()
    print(f"Using T={bt}, ms={bms}")
    
    combos = [(True,True,True,True,True),(False,False,False,False,False),
              (True,True,False,False,False),(False,False,True,True,True)]
    names = ['all_on','all_off','3d_only','slice_only']
    flags = ['3d_close','3d_fill','slice_close','slice_fill','slice_open']
    
    results = []; t0 = time.time(); vc = 0
    for fold_idx in [0, 1]:
        swa_dir = cfg.PRED_DIR / f"fold{fold_idx}_swa"
        if not swa_dir.exists(): continue
        for npy in sorted(swa_dir.glob("*.npy")):
            vid = npy.stem; gt_path = labels_dir / f"{vid}.tif"
            if not gt_path.exists(): continue
            prob = np.load(str(npy)).astype(np.float32)
            gt = tifffile.imread(str(gt_path)).astype(np.uint8)
            for ci, c in enumerate(combos):
                mask = postprocess(prob, bt, bms, *c)
                sc = compute_competition_score(mask, gt, 2)
                row = {'fold':fold_idx,'vol_id':vid,'threshold':bt,'min_size':bms}
                for fi,f in enumerate(flags): row[f] = c[fi]
                row.update(sc); results.append(row); del mask
            del prob, gt; gc.collect(); vc += 1
            print(f"  [{vc}] f{fold_idx} {vid}")
    
    df = pd.DataFrame(results); df.to_csv(cfg.OUTPUT_DIR/"postprocess_ablation.csv", index=False)
    print(f"\nPP RESULTS:")
    for ci, nm in enumerate(names):
        c = combos[ci]
        sub = df
        for fi,f in enumerate(flags): sub = sub[sub[f]==c[fi]]
        print(f"  {nm:<12}: LB={sub['lb_score'].mean():.4f}")
    return df

pp_results = quick_pp()


In [ ]:
# =============================================================================
# CELL 7: MODEL COMPARISON + RESULTS SUMMARY
# =============================================================================
def show_results():
    # Model comparison
    mc = pd.read_csv(cfg.OUTPUT_DIR/"model_comparison.csv")
    ms = mc.groupby('model')['lb_score'].mean().sort_values(ascending=False)
    print("MODEL RANKING:")
    for m,lb in ms.items(): print(f"  {m:<20s}: {lb:.4f}{'  ★' if 'swa' in str(m) else ''}")
    
    # Best config
    bt, bms = 0.50, 50
    if Path(cfg.OUTPUT_DIR/"threshold_sweep.csv").exists():
        ts = pd.read_csv(cfg.OUTPUT_DIR/"threshold_sweep.csv")
        cm = ts.groupby(['threshold','min_size'])['lb_score'].mean()
        bt, bms = cm.idxmax()
    
    best_pp = 'all_on'
    if Path(cfg.OUTPUT_DIR/"postprocess_ablation.csv").exists():
        pp = pd.read_csv(cfg.OUTPUT_DIR/"postprocess_ablation.csv")
        flags = ['3d_close','3d_fill','slice_close','slice_fill','slice_open']
        pp['combo'] = pp[flags].apply(lambda r: tuple(r), axis=1).astype(str)
        best_pp = pp.groupby('combo')['lb_score'].mean().idxmax()
    
    config = {'threshold':bt,'min_component_size':bms,'overlap':0.5,'use_swa':True,
              'do_3d_close':True,'do_3d_fill':True,'do_slice_close':True,'do_slice_fill':True,'do_slice_open':True}
    
    print(f"\n{'='*50}")
    print(f"OPTIMAL CONFIG FOR SUBMISSION:")
    print(f"  THRESHOLD = {bt}")
    print(f"  MIN_COMPONENT_SIZE = {bms}")
    print(f"  OVERLAP = 0.5")
    print(f"  USE_SWA = True (all 8 models)")
    print(f"  Best PP: {best_pp}")
    print(f"  Expected LB ≈ {ms.iloc[0]:.4f}")
    
    with open(cfg.OUTPUT_DIR/"optimal_config.json",'w') as f: json.dump(config,f,indent=2)
    print(f"\n  Saved: {cfg.OUTPUT_DIR/'optimal_config.json'}")

show_results()
